# **ESMC를 사용한 임베딩 추출**

2025 겨울 URP / 문서연, 김대현, 권효재


---


기존의 sequence based Protein Language Model은 한계가 있다고 판단, sturcture과 functioin도 함께 학습한 Multimodal Model인 ESM3를 선택

단, 우리가 진행할 전이학습은 input에 sequence만 넣기 때문에, ESM3에서 sequence embedding 추출에 특화된 모델인 ESMC로 진행하기로 함.


---


개선/공부 필요(2026_01_18):


*   transformer(huggingface) 라이브러리 의존: 의존할거면 아예 pipeline으로 더 간단하게 작성할 수도 있을 거라고 생각. 라이브러리 사용하지 않고 구현하는 방식도 공부해보고 싶음.
*   TOKENIZER 관련: 모델마다 토크나이저 설정이 달라 모델 변경 시에 추가 설정이 필요해짐. 호환성을 위해 개선 가능할거라고 생각
*   MAX_LEN 관련: 기존 모델들은 학습 시퀀스 길이에 제한을 둠. 관련 공부가 필요
*   BATCH_SIZE 관련: 현재 코드가 배치 사이즈 1 이상인 경우에 대해 잘 대응하지 못함. 특히 마지막으로 코드를 돌렸을 때 배치 사이즈가 다른 파일을 저장하지 못하는 문제가 있었음. 코드 수정과 동적 패딩과 배치 사이즈 관련에서 공부가 필요.
*   TOKENIZER 관련: 토큰화 시에 시퀀스 앞뒤로 특수토큰 추가하는데(`<cls>, <eos>`), 이후 학습에 방해가 되니 삭제하라는 AI의 의견이 있음. 그러나, 시작과 끝을 명시한다는 점에서 서열의 시작/끝 또한 매우 중요한 정보이니 남겨야 한다는 생각과, 동시에 `<cls>` 토큰의 경우 그 서열 전체의 대표 정보를 반환한다는 내용이 있는데, 그렇다면 서열의 맥락 학습에 오히려 해당 토큰이 비이상적으로 큰 가중치를 받게 되어 문제를 일으킬 수 있겠다는 생각도 듦. 근데 결국 pooling으로 한 벡터로 합치려고 하는 입장에서 그것조차 학습 파라미터 조정으로 해결될 수 있는 부분이 아닌가 하는 생각도 들고.. 학습과 질문 필요
*   RAM 관련: 반복문 학습을 돌리는 중 RAM이 커서 터지는 경우가 발생한 적 있음. 특히 지금 방식은 list에 담는 방식이라 cpu RAM에 의존적임. 이를 해결하기 위해 h5py 라이브러리를 사용하여 저장을 시도했으나, 굉장히 오래 걸리고, dataset으로 저장 시에 넘파이 배열로 저장해야하기 때문에 결국 차원을 맞춰줘야 하는 문제가 발생함. 결국 차원을 맞춰준다면 배치 사이즈도 큰 게 나을 것 같아 1500 길이로 패딩하여 저장하니 총 용량이 140gb가량 나옴. 이에 관련해서도 더 나은 라이브러리를 찾아보거나 질의가 필요

In [1]:
#필요 라이브러리 설치 및 import
!pip install transformers tqdm torch numpy pandas

from transformers import AutoModelForMaskedLM, AutoTokenizer

import os
import tqdm
import torch
import numpy as np
import pandas as pd
import gc

In [2]:
#<저장소 설정>

#Colab 환경
from google.colab import drive
drive.mount('/content/drive')
SAVE_PATH = '/content/drive/MyDrive/Github/Mprotein_hydrophobic/ESMC_embedding'

#로컬 환경
#SAVE_PATH = './ESMC_embedding'

os.makedirs(SAVE_PATH, exist_ok=True)

Mounted at /content/drive


In [3]:
#<기본 변수 설정>

#사용할 모델의 Huggingface id 설정
MODEL_ID = 'Synthyra/ESMplusplus_large'

#gpu device 설정
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

#모델&토크나이저 호출
MODEL_PRETRAINED = AutoModelForMaskedLM.from_pretrained(MODEL_ID, trust_remote_code=True).to(DEVICE).eval()
TOKENIZER = MODEL_PRETRAINED.tokenizer

#csv에서 데이터 호출, 시퀀스 길이가 1498(토큰추가 고려) 넘으면 그 행 drop
#(ESMC는 최대 길이 제한이 존재하지 않기 때문에 사실 MAX_LEN은 사용하지 않음)
MAX_LEN = 1500

#또한, 인풋 sequence의 길이가 매우 변칙적이기 때문에, 배치 사이즈를 키워 저장하는 것은 용량적으로 큰 무리가 있음
#따라서 그냥 배치 사이즈는 1로 진행, 이게 동적 패딩이라고 부르는 게 맞는지는 확실치 않음
BATCH_SIZE = 1

#이후 분류에 정답이 될 라벨 정보를 (1,0,0,0)과 같은 형식으로 추출, 텐서로 변환하여 파티션별로 저장하기 위해 라벨명(column)을 리스트로 저장함
TARGET_LABELS = ['Peripheral', 'Transmembrane', 'LipidAnchor', 'Soluble']

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/771 [00:00<?, ?B/s]

modeling_esm_plusplus.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/Synthyra/ESMplusplus_large:
- modeling_esm_plusplus.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/2.30G [00:00<?, ?B/s]

In [4]:
#<input 데이터 호출 및 전처리>
df_raw = pd.read_csv('https://github.com/Rainbowbarkbark/Mprotein_hydrophobic/blob/main/Swissprot_Membrane_Train_Validation_dataset.csv?raw=true')

#원래는 padding을 위한 차원 길이 맞추기 + ESM2의 길이 제한으로 인해 적정 길이 이상의 데이터를 날림
#그러나 배치 사이즈도 1이고, ESMC는 input sequence의 길이 제한이 없기 때문에 추가 처리를 하지 않음
#길이제한이 있는 모델과 그렇지 않은 모델의 차이가 무엇인지 이후 확인 필요!
#df_max_len = df_raw[df_raw['Sequence'].str.len() <= (MAX_LEN-2)].reset_index(drop=True)

df_max_len = df_raw.copy()

In [16]:
#<임베딩 추출 함수 선언>
#이후에 패딩 사이즈에 따라서 attention mask 빼오는 연산 유뮤도 달라지게 하면 더 좋을듯
#근데 이게 유의미하게 속도에 차이를 줄지는 잘 모르겠음

def embedding_extract(model, tokenizer, per_sequence, device, max_length):

    #인풋으로 받는 sequence가 2개 이상일 경우 최대 길이에 맞추어 자동으로 <pad>토큰 추가하고, pytorch 텐서로 리턴
    #padding='max_length'로 설정할 경우, sequence의 길이와 상관 없이 길이에 맞추어 padding 해줌
    tokenized = tokenizer(
        per_sequence,
        padding = True,
        return_tensors ='pt',
        #max_length = max_length
        )

    #토큰화된 결과는 'input_ids', 'attention_mask'가 key인 딕셔너리 형태, 어텐션 마스크 미리 가져오기
    #그러나 이것도 배치를 1로 하면서 의미 없어짐
    attention_mask = tokenized['attention_mask']

    #GPU로 옮겨두기
    tokenized = tokenized.to(device)

    #역전파할거 아니니까 그래디언트 연산 끄고, 임베딩 추출
    #output_hidden_states를 True로 해야 나중에 hidden_states[레이어층수]로 히든레이어의 임베딩을 빼올 수 있음
    #ESM++은 last_hidden_state를 지원하기 때문에 사실 True 켜둘 필요 없긴 함
    with torch.no_grad():
        output = model(**tokenized, output_hidden_states=True)

    #결과물을 cpu로 가져와서 저장, 리턴
    embeddings = output.hidden_states[-1].cpu()     #last_hidden_state / hidden_states[-1]
    attention_mask = attention_mask.cpu()
    return embeddings, attention_mask

#작동 확인
#embeddings, _ = embedding_extract(MODEL_PRETRAINED, TOKENIZER, df_raw['Sequence'][0], DEVICE, MAX_LEN)
#print(embeddings.shape)

In [ ]:
#<임베딩 추출 진행>
#코드가 배치 사이즈에 따라 주석을 수정해야 하는 경우가 많음 추후 수정할 수 있었으면 좋겠음
for k in range(4):

  #중요한건 아닌데 파일 저장 이름 설정
  #k번째 파티션별로 파일을 따로 저장해줌
  SAVE_PATH_TARGET = os.path.join(SAVE_PATH, f'target_part_{k}.pt')
  SAVE_PATH_EMBEDDINGS = os.path.join(SAVE_PATH, f'embeddings_part_{k}.pt')

  #파티션 k로만 이루어진 데이터프레임 df_part_k 복제
  df_part_k = df_max_len[df_max_len['Partition'] == k].copy().reset_index(drop=True)

  #df_part_k의 target(정답 labels)(array), sequences(array > list) 저장
  df_target_k = df_part_k[TARGET_LABELS].values
  df_sequences_k = df_part_k['Sequence'].values.tolist()

  #pytorch 텐서로 변환 후 저장
  df_target_k = torch.from_numpy(df_target_k).long()
  torch.save(df_target_k, SAVE_PATH_TARGET)

  #임베딩 추출을 위해 빈 리스트 생성
  tmp_embeddings = []
  #tmp_attmasks = []                                                                                      #어텐션 마스크 필요 시 주석 해제

  #제일 중요한 반복문
  #tqdm으로 반복문 진행도 확인 가능하게 하고
  #인풋 시퀀스를 배치 사이즈에 맞게 슬라이싱함
  #배치 사이즈가 1이니까 앞에 필요없는 차원 하나 날림
  for i in tqdm.tqdm(range(0, len(df_sequences_k), BATCH_SIZE)):
      batch_sequences = df_sequences_k[i:i+BATCH_SIZE]
      embeddings, _ = embedding_extract(MODEL_PRETRAINED, TOKENIZER, batch_sequences, DEVICE, MAX_LEN)    #어텐션 마스크 필요 시 _ > attention_masks
      embeddings = embeddings.squeeze(0)                                                                  #어텐션 마스크 필요 시 주석 처리 (의미 없어짐)
      tmp_embeddings.append(embeddings)
      #tmp_attmasks.append(attention_masks)                                                               #어텐션 마스크 필요 시 주석 해제

  #임베딩 텐서로 저장해줌
  torch.save(tmp_embeddings, SAVE_PATH_EMBEDDINGS)

  #파일 다 저장했으면 GPU RAM 관리를 위해 함수 없애고 캐시 삭제
  del tmp_embeddings, df_part_k, df_target_k, df_sequences_k
  #del tmp_attmasksq                                                                                      #어텐션 마스크 필요 시 주석 해제
  gc.collect()
  torch.cuda.empty_cache()

100%|██████████| 6175/6175 [06:45<00:00, 15.23it/s]


In [ ]:
!nvidia-smi
'''
#GPU RAM 정리
gc.collect()
torch.cuda.empty_cache()

try:
    del MODEL_PRETRAINED
    del TOKENIZER
    del embeddings
except NameError:
    pass'''

Fri Jan 16 17:08:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   74C    P0             34W /   70W |     150MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [17]:
'''
a = df_max_len['Sequence'].tolist()
t = TOKENIZER(a[0], return_tensors = 'pt')
#t = t.to(DEVICE)
#print(t)
embedding_extract(MODEL_PRETRAINED, TOKENIZER, a[0], DEVICE, MAX_LEN)'''

(tensor([[[ -12.8587,  -62.4924,  -13.6469,  ...,   -2.4262,   71.8056,
            155.2066],
          [ 198.8174,  -69.1296,  -54.4747,  ..., -136.7207,  -34.6456,
             37.6476],
          [ 195.3465,  -31.6195, -101.6760,  ..., -118.6017,  -60.3891,
             47.4117],
          ...,
          [  16.7748, -123.3613, -140.1055,  ..., -209.6243,  203.4697,
           -112.3972],
          [  88.4894, -291.8088,  -22.5020,  ..., -155.5294,   12.8213,
           -269.0920],
          [  55.5003,  -79.6198,  -46.5753,  ...,  -88.6954,   31.9706,
            -39.8250]]]),
 tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
  

In [ ]:
'''#저장할 파일 만들기
import h5py

#1.저장할 파일명
filename = 'ESMC_embedding.h5'
FILE_PATH = os.path.join(SAVE_PATH, filename)
print(FILE_PATH)
#2. partition, label 정보는 바로 저장, 어텐션 마스크와 임베딩은 빈 파일로 생성
with h5py.File(FILE_PATH, 'w') as f:
    f.create_dataset('partitions', data=df_partitions)
    f.create_dataset('labels', data=df_labels)
    f.create_dataset('embeddings',
                     shape=(0, MAX_LEN, 1153), #이거 hiddendim사이즈맞춰줘야함!!
                     maxshape = (None, MAX_LEN, 1153),
                     chunks = True,
                     dtype='float32')
    f.create_dataset('attention_masks',
                     shape=(0, MAX_LEN),
                     maxshape=(None, MAX_LEN),
                     chunks=True,
                     dtype='int8')

for i in tqdm.tqdm(range(0, len(df_sequences_raw), BATCH_SIZE)):
    batch_sequences = df_sequences_raw[i:i+BATCH_SIZE]
    embeddings = embedding_extract(model, tokenizer, batch_sequences, device, MAX_LEN).embeddings

    with h5py.File(FILE_PATH, 'a') as f:
        #현재 데이터셋 크기
        curr_size = f['embeddings'].shape[0]

        #데이터셋 크기 늘리기
        f['embeddings'].resize(curr_size + BATCH_SIZE, axis=0)
        f['attention_masks'].resize(curr_size + BATCH_SIZE, axis=0)

        #마지막에 20개 배치 만들지 못하는 경우엔 남은 갯수만 저장됨
        num_items = len(embeddings)

        #새로운 데이터 추가
        f['embeddings'][curr_size:curr_size + num_items] = embeddings.astype('float32')
        f['attention_masks'][curr_size:curr_size + num_items] = attention_masks.astype('int8')'''